# Nonstationary GP

入力位置によって latent function の smoothness が変わる場合に Gibbs kernel を使います。


## このNotebookで学ぶこと

- 定常カーネルと非定常カーネルの違い
- Gibbs kernelで入力位置ごとのlengthscaleを表現する方法
- `make_mll()` とBoTorchの `fit_gpytorch_mll()` による学習
- 事後平均・不確実性・局所lengthscaleの可視化
- **事後平均による有限候補選択**と**qLogEIの最適化**の違い
- Mixed・MultiTask・Kroneckerモデルの入力構造

このNotebookでは候補提案後に合成目的関数を評価します。観測を追加して再学習する完全な反復最適化ループは対象外です。

## モデル構築


### 定常と非定常の違い

一般的な定常カーネルは、2点間の共分散が主としてその距離に依存します。一方、非定常GPでは入力位置によって関数の滑らかさが変わることを表現できます。

例えば製造工程で、安定な領域では応答がゆっくり変化し、相転移や閾値付近では急激に変化する場合です。非定常モデルはそのような局所的な変化に対応する選択肢です。

In [ ]:
from botorch.fit import fit_gpytorch_mll
import torch
from robotorchan.models import NonstationarySingleTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X = torch.linspace(0, 1, 16).unsqueeze(-1)
Y = torch.where(X < 0.5, torch.sin(2 * torch.pi * X), torch.sin(8 * torch.pi * X))
model = NonstationarySingleTaskGP(X, Y)
fit_gpytorch_mll(model.make_mll())


### 合成データと学習

$x<0.5$ では低周波の正弦波、$x\ge 0.5$ では高周波の正弦波を使います。**入力位置で変化の速さが異なる**ことがポイントです。

`fit_gpytorch_mll(model.make_mll())` はモデルの周辺対数尤度を用いてハイパーパラメータを学習します。ここではrobotorchanのモデル構築APIとBoTorchの既存学習APIを組み合わせています。

## 局所 lengthscale


In [ ]:
test_X = torch.linspace(0, 1, 5).unsqueeze(-1)
print(model.local_lengthscale(test_X))
print(model.posterior(test_X).mean.squeeze(-1))


### 局所lengthscaleの読み方

`local_lengthscale(x)` は入力位置ごとの長さ尺度を確認するAPIです。一般に、小さなlengthscaleは近い入力でも応答が変化しやすいことを、大きなlengthscaleは滑らかな変化を表す傾向があります。

ただし、**学習されたlengthscaleが真の変化点を正確に特定するとは限りません**。学習点数・初期値・モデル仮定の影響を受けます。

### 追加実装例：予測分布と局所lengthscaleを可視化

真の関数、観測点、GPの予測平均、潜在関数の近似95%事後区間を同じ図で比較します。別の図で局所lengthscaleの位置依存性を確認します。

ここでの95%区間は**潜在関数の事後分布**に基づくもので、測定ノイズを含む観測予測区間ではありません。

In [ ]:
import matplotlib.pyplot as plt

plot_X = torch.linspace(0.0, 1.0, 201).unsqueeze(-1)
model.eval()
with torch.no_grad():
    plot_post = model.posterior(plot_X)
    plot_mean = plot_post.mean.squeeze(-1)
    plot_std = plot_post.variance.clamp_min(0).sqrt().squeeze(-1)
    plot_lengthscale = model.local_lengthscale(plot_X).reshape(-1)
true_Y = torch.where(plot_X < 0.5, torch.sin(2 * torch.pi * plot_X),
                     torch.sin(8 * torch.pi * plot_X))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(plot_X.squeeze(-1), true_Y.squeeze(-1), "--", label="true function")
ax.scatter(X.squeeze(-1), Y.squeeze(-1), s=20, label="observations")
ax.plot(plot_X.squeeze(-1), plot_mean, label="posterior mean")
ax.fill_between(plot_X.squeeze(-1), plot_mean - 1.96 * plot_std,
                plot_mean + 1.96 * plot_std, alpha=0.2,
                label="approx. 95% latent interval")
ax.axvline(0.5, linestyle=":", label="frequency change")
ax.legend(fontsize=8)
ax.set_xlabel("x")
ax.set_ylabel("response")
plt.show()
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(plot_X.squeeze(-1), plot_lengthscale)
ax.axvline(0.5, linestyle=":")
ax.set_xlabel("x")
ax.set_ylabel("local lengthscale")
plt.show()

## Nonstationary candidate proposal

非定常カーネルによる事後平均から候補を選び、元の目的関数で評価します。


In [ ]:
candidate_grid = torch.linspace(0, 1, 101).unsqueeze(-1)
model.eval()
with torch.no_grad():
    scores = model.posterior(candidate_grid).mean.squeeze(-1)
candidate = candidate_grid[scores.argmax().item():scores.argmax().item() + 1]
candidate_Y = torch.where(
    candidate < 0.5,
    torch.sin(2 * torch.pi * candidate),
    torch.sin(8 * torch.pi * candidate),
)
assert candidate.shape == (1, 1)
print("candidate:", candidate, "evaluated objective:", candidate_Y)


### 有限候補集合による選択

既存の候補提案は101点のグリッド上で**事後平均が最大の点**を選びます。これは探索不確実性を直接利用しないgreedyな活用です。

以下では同じ学習済みモデルに対して、BoTorchの `qLogExpectedImprovement` と `optimize_acqf` を使った候補最適化も実行します。候補が異なっても不具合とは限りません。

In [ ]:
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf

acqf = qLogExpectedImprovement(model=model, best_f=Y.max())
optimized_candidate, acq_value = optimize_acqf(
    acq_function=acqf,
    bounds=torch.tensor([[0.0], [1.0]]),
    q=1,
    num_restarts=3,
    raw_samples=32,
)
optimized_observation = torch.where(
    optimized_candidate < 0.5,
    torch.sin(2 * torch.pi * optimized_candidate),
    torch.sin(8 * torch.pi * optimized_candidate),
)
print("qLogEI candidate:", optimized_candidate)
print("qLogEI value:", acq_value)
print("evaluated objective:", optimized_observation)

## Mixed / multi-task / Kroneckerの候補提案

単一出力・long-format multi-task・block-design Kroneckerの4モデルを追加します。有限候補プールの事後平均を比較し、カテゴリとtaskの構造を保持して目的関数を評価します。これは獲得関数の連続最適化ではありません。


### Mixed・MultiTask・Kroneckerの違い

- **Mixed**：連続設計変数とカテゴリ設計変数を同時に扱います
- **MultiTask（long-format）**：タスクIDを入力に含め、異なるタスクの観測を行方向に積み重ねます
- **Kronecker MultiTask（block-design）**：共通設計点に対して複数タスクの応答を列方向に並べます

入力の形が異なるため、候補生成時のPosterior呼び出しも同一ではありません。以下は既存モデルの学習・候補評価を再利用する例です。

In [ ]:
from robotorchan.models import (
    MixedNonstationaryMultiTaskGP,
    MixedNonstationarySingleTaskGP,
    NonstationaryKroneckerMultiTaskGP,
    NonstationaryMultiTaskGP,
)

def nonstationary_target(x):
    return torch.where(
        x < 0.5, torch.sin(2 * torch.pi * x),
        torch.sin(8 * torch.pi * x),
    )


design_x = torch.linspace(0.05, 0.95, 12).unsqueeze(-1)
category = torch.arange(len(design_x)).remainder(2).to(design_x).unsqueeze(-1)
mixed_design = torch.cat([design_x, category], dim=-1)
base_y = nonstationary_target(design_x)
mixed_y = base_y + 0.1 * category
task_y = base_y + 0.2
mixed_task_y = mixed_y + 0.2

long_X = torch.cat(
    [
        torch.cat([design_x, torch.zeros(len(design_x), 1)], dim=-1),
        torch.cat([design_x, torch.ones(len(design_x), 1)], dim=-1),
    ],
    dim=0,
)
long_Y = torch.cat([base_y, task_y], dim=0)
mixed_long_X = torch.cat(
    [
        torch.cat([mixed_design, torch.zeros(len(design_x), 1)], dim=-1),
        torch.cat([mixed_design, torch.ones(len(design_x), 1)], dim=-1),
    ],
    dim=0,
)
mixed_long_Y = torch.cat([mixed_y, mixed_task_y], dim=0)
block_Y = torch.cat([base_y, task_y], dim=-1)

nonstationary_models = [
    (
        "Mixed single-task",
        MixedNonstationarySingleTaskGP(
            mixed_design, mixed_y, cat_dims=[1]
        ),
    ),
    (
        "Multi-task",
        NonstationaryMultiTaskGP(
            long_X, long_Y, task_feature=1
        ),
    ),
    (
        "Mixed multi-task",
        MixedNonstationaryMultiTaskGP(
            mixed_long_X, mixed_long_Y, task_feature=2, cat_dims=[1]
        ),
    ),
    (
        "Kronecker multi-task",
        NonstationaryKroneckerMultiTaskGP(design_x, block_Y),
    ),
]
for name, surrogate in nonstationary_models:
    fit_gpytorch_mll(surrogate.make_mll())
    surrogate.eval()
    print(name, "trained")


### 各モデルの学習確認

4モデルそれぞれに対し、`fit_gpytorch_mll(surrogate.make_mll())` で学習を実行します。`cat_dims=[1]` は2列目がカテゴリ設計変数、`task_feature` はlong-formatのタスクID列を示します。

この後の候補提案は**有限候補集合の事後平均ランキング**です。上のqLogEIによる連続最適化とは異なります。

In [ ]:
candidate_x = torch.linspace(0.0, 1.0, 101).unsqueeze(-1)
candidate_category = torch.cat(
    [
        torch.cat([candidate_x, torch.zeros(len(candidate_x), 1)], dim=-1),
        torch.cat([candidate_x, torch.ones(len(candidate_x), 1)], dim=-1),
    ],
    dim=0,
)
for name, surrogate in nonstationary_models:
    if name.startswith("Mixed"):
        pool = candidate_category
    else:
        pool = candidate_x
    with torch.no_grad():
        if name.endswith("multi-task") and name != "Kronecker multi-task":
            posterior = surrogate.posterior(pool, output_indices=[1])
        else:
            posterior = surrogate.posterior(pool)
        scores = posterior.mean
    if name == "Kronecker multi-task":
        scores = scores[..., 1]
    else:
        scores = scores.squeeze(-1)
    selected = scores.argmax().item()
    candidate = pool[selected:selected + 1]
    observed = nonstationary_target(candidate[:, :1])
    if name.startswith("Mixed"):
        observed = observed + 0.1 * candidate[:, 1:2]
        assert candidate[0, 1].item() in (0.0, 1.0)
    if "multi-task" in name:
        observed = observed + 0.2
    assert candidate.shape == (1, pool.shape[-1])
    assert torch.isfinite(observed).all()
    print(name, "candidate:", candidate, "objective:", observed)


### Mixed SingleTaskの獲得関数最適化

Mixedの単一出力モデルでは、既存のBoTorch `optimize_acqf_mixed` を使用できます。カテゴリ列を `fixed_features_list` で列挙し、連続列は各カテゴリについて最適化します。ここではqLogEIを使用し、事後平均による有限プール選択との違いを確認します。

MultiTask・Kroneckerモデルには同じ処理を機械的に適用しません。タスクの固定・対象出力の選択を考慮する必要があるため、このNotebookでは既存の候補プール方式を維持します。

In [ ]:
from botorch.optim.optimize import optimize_acqf_mixed

mixed_single_model = nonstationary_models[0][1]
mixed_single_model.eval()
mixed_acqf = qLogExpectedImprovement(
    model=mixed_single_model,
    best_f=mixed_y.max(),
)
mixed_bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])
mixed_optimized_candidate, mixed_acq_value = optimize_acqf_mixed(
    acq_function=mixed_acqf,
    bounds=mixed_bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
    fixed_features_list=[{1: 0.0}, {1: 1.0}],
)
assert mixed_optimized_candidate.shape == (1, 2)
assert mixed_optimized_candidate[0, 1].item() in (0.0, 1.0)
mixed_optimized_y = (
    nonstationary_target(mixed_optimized_candidate[:, :1])
    + 0.1 * mixed_optimized_candidate[:, 1:2]
)
print("Mixed qLogEI candidate:", mixed_optimized_candidate)
print("Mixed qLogEI value:", mixed_acq_value)
print("evaluated objective:", mixed_optimized_y)

### MultiTask候補の評価方法

long-formatモデルでは `output_indices=[1]` により対象タスクを指定しています。Kroneckerモデルでは複数タスク出力のうち対象列を選択します。

候補の目的関数評価では、カテゴリによるオフセットとタスクによるオフセットを反映します。候補の選択時にどのタスクの事後平均を最大化しているかを意識してください。

## まとめ

非定常GPは、入力領域ごとに滑らかさが異なる応答を扱うためのモデルです。

| 確認したいこと | 使用するAPI |
|---|---|
| 学習 | `fit_gpytorch_mll(model.make_mll())` |
| 局所的な滑らかさ | `local_lengthscale(X)` |
| 予測平均・不確実性 | `posterior(X)` |
| 簡単な候補選択 | 有限プールの事後平均最大化 |
| 獲得関数に基づく候補最適化 | `qLogExpectedImprovement` + `optimize_acqf` |

**留意点：** Mixed SingleTaskでは `optimize_acqf_mixed` によるqLogEI最適化も実装しました。MultiTask・Mixed MultiTask・Kronecker版は有限候補選択までです。これらに獲得関数最適化を適用する場合は、タスク指定・出力形状に合った方法を別途確認します。